In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
from tsfm_lens.utils import (
    apply_custom_style,
    load_dyst_samples,
    set_seed,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../../figs", "ablations_chronos", "examples")
os.makedirs(plot_save_dir, exist_ok=True)


In [ ]:
pipeline = CircuitLensChronos.from_pretrained("amazon/chronos-t5-base", device_map=device, torch_dtype=torch.bfloat16)
num_layers = pipeline.num_layers
num_heads = pipeline.num_heads
print(f"num_layers: {num_layers}, num_heads: {num_heads}")

In [ ]:
model_name = "chronos-t5-base"
config_key = "rf8_sl64"

results_dir = "/work/results/rrt_induction_scores"
with open(os.path.join(results_dir, model_name, config_key, "center_scores.pkl"), "rb") as f:
    center_scores = pickle.load(f)
    center_scores_mean, center_scores_std = center_scores["mean"], center_scores["std"]

with open(os.path.join(results_dir, model_name, config_key, "right_scores.pkl"), "rb") as f:
    right_scores = pickle.load(f)
    right_scores_mean, right_scores_std = right_scores["mean"], right_scores["std"]

with open(os.path.join(results_dir, model_name, config_key, "expected_token_attribution.pkl"), "rb") as f:
    expected_token_attribution = pickle.load(f)
    
with open(os.path.join(results_dir, model_name, config_key, "rrt_vars.pkl"), "rb") as f:
    rrt_vars = pickle.load(f)

In [ ]:
expected_token_attribution.keys()

In [ ]:
def find_max_induction_score(induction_scores: torch.Tensor, k: int | None = None) -> list[tuple[int, int]]:
    """
        Find the indices of the top k highest induction scores in the matrix.

        Args:
            induction_scores: torch tensor of induction scores
            k: number of top scores to return
    w
        Returns:
            List of (layer, head) tuples corresponding to the top k induction scores
    """
    # Flatten the tensor to find the top k values
    flattened = induction_scores.flatten()

    # Get the indices of the top k values
    if k is None or k > flattened.numel():
        k = flattened.numel()

    top_k_values, top_k_indices = torch.topk(flattened, k)

    # Convert flat indices back to 2D indices (layer, head)
    num_heads = induction_scores.shape[1]
    layer_indices = top_k_indices // num_heads
    head_indices = top_k_indices % num_heads

    # Create list of (layer, head) tuples
    top_k_heads = [(int(layer_indices[i].item()), int(head_indices[i].item())) for i in range(k)]

    return top_k_heads


In [ ]:
maxed_ind_scores = torch.maximum(center_scores_mean, right_scores_mean)

In [ ]:
split_name = "base40"

split_dir = os.path.join(DATA_DIR, split_name)
system_name = "Thomas"
# ensemble = make_ensemble_from_arrow_dir(split_dir, dyst_names_lst=[system_name], num_samples=1)

context_length = 512
context_start_time = 0
context_end_time = context_start_time + context_length

# Prepare input series
sample_idx = 0
selected_dim = 0
dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :]).unsqueeze(0)
context = dyst_coords[:, context_start_time:context_end_time]

print(context.shape)

prediction_length = 512
future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
print(f"future_vals shape: {future_vals.shape}")


In [ ]:
pipeline.remove_all_hooks()

# layers_to_ablate = []
# layers_to_ablate = [7, 8, 9]
layers_to_ablate = find_max_induction_score(right_scores_mean, 144)
# layers_to_ablate = layers_to_ablate[139:]
layers_to_ablate = []
# layers_to_ablate = [2, 3, 4, 5]

######### Attention Head ablations #########
# pipeline.add_head_ablation_hooks(
#     layers_to_ablate,
#     attention_type="ca",
# )
# pipeline.add_head_ablation_hooks(
#     layers_to_ablate,
#     attention_type="sa",
# )

######### MLP ablations #########
# pipeline.add_mlp_ablation_hooks(layers_to_ablate, ablation_method="zero")
# pipeline.add_mlp_ablation_hooks([1, 2], ablation_method="zero")

In [ ]:
layers_without_ca_heads = list(pipeline.ca_head_ablation_handles.keys())
layers_without_sa_heads = list(pipeline.sa_head_ablation_handles.keys())
layers_without_mlps = list(pipeline.mlp_ablation_handles.keys())

layers_without_heads = layers_without_ca_heads

if layers_without_heads and layers_without_mlps:
    if layers_without_heads != layers_without_mlps:
        # raise NotImplementedError("Zero-ablation of heads and MLPs in different layers is messier, save for later")
        pass
    ablations_summary_str_title = f"Zero-Ablation: Heads and MLPs in Layers {layers_without_heads}"
    ablations_summary_str = f"za_heads_mlps_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_heads:
    ablations_summary_str_title = f"Zero-Ablation: Heads in Layers {layers_without_heads}"
    ablations_summary_str = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_mlps:
    ablations_summary_str_title = f"Zero-Ablation: MLPs in Layers {layers_without_mlps}"
    ablations_summary_str = f"za_mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
else:
    ablations_summary_str = ablations_summary_str_title = None

print(ablations_summary_str)
print(ablations_summary_str_title)


In [ ]:
rseed = 123
set_seed(rseed)

num_samples = 10
do_sample = True if num_samples > 1 else False

# pred, _ = pipeline.predict_with_full_outputs(  # type: ignore
#     context=context,
#     prediction_length=prediction_length,
#     num_samples=num_samples,
#     do_sample=do_sample,  # greedy decoding if num_samples == 1
#     use_cache=True,
#     return_dict_in_generate=True,
#     output_scores=True,
#     limit_prediction_length=False,
# )

pred = pipeline.predict(  # type: ignore
    inputs=context,
    prediction_length=prediction_length,
    num_samples=num_samples,
    limit_prediction_length=False,
)

In [ ]:
# Prepare context and predictions
context_np = context.squeeze().cpu().numpy()
context_timesteps = np.arange(context_np.shape[-1])
future_vals_np = future_vals.squeeze()
future_timesteps = np.arange(context_np.shape[-1], context_np.shape[-1] + prediction_length)
print(f"context_np shape: {context_np.shape}")
print(f"future_vals_np shape: {future_vals_np.shape}")
print(f"length of context_timesteps: {len(context_timesteps)}")
print(f"length of future_timesteps: {len(future_timesteps)}")

In [ ]:
preds = pred.squeeze()
if preds.ndim == 1:
    preds = preds[None, :]

fig, ax = plt.subplots(figsize=(6, 2))

# Plot context, ground truth and predictions
ax.plot(context_timesteps, context_np, color="black", linewidth=1, label="Context")
ax.plot(future_timesteps, future_vals_np, color="black", linewidth=1, linestyle="--", label="Ground Truth")
for i in range(len(preds)):
    ax.plot(future_timesteps, preds[i], color=DEFAULT_COLORS[i], linewidth=1, alpha=0.5)
ax.plot(future_timesteps, np.median(preds, axis=0), color="tab:blue", linewidth=2, label="Median")

ax.set_xlabel("Timestep", fontweight="bold")

# Save plot
save_fname = (
    f"predictions_{system_name}_dim{selected_dim}_start{context_start_time}_sub{1}"
    if split_name == "base40"
    else f"predictions_{system_name}"
)
save_path = os.path.join(plot_save_dir, "zero_ablations", f"{ablations_summary_str}_{save_fname}.pdf")

if ablations_summary_str_title:
    print(ablations_summary_str_title)
print(f"Saving to {save_path}")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.tight_layout()
fig.savefig(save_path, bbox_inches="tight")